In [3]:
!pip install -q transformers accelerate openpyxl bitsandbytes

In [4]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import pandas as pd

import pandas as pd

file_path = "/content/drive/MyDrive/Austrian Tax Law LLM Prompt Assignment.xlsx"

df = pd.read_excel(file_path, sheet_name="Dataset")
df = df[["id", "prompt", "correct_answer", "sources"]].dropna(subset=["prompt"]).reset_index(drop=True)

print("Questions:", len(df))
df.head(3)

Questions: 645


,id,prompt,correct_answer,sources
0,CORP-TAX-001,Was ist die steuerliche Bemessungsgrundlage fü...,"Das Einkommen, das der unbeschränkt Steuerpfli...",§ 7 Abs. 1 KStG 1988
1,CORP-TAX-002,"Welche steuerlichen Konsequenzen hat es, wenn ...",Die nicht verrechneten Zinsen stellen eine ver...,§ 7 Abs. 2/3 KStG 1988; § 8 Abs. 2 KStG 1988; ...
2,CORP-TAX-003,"Welche Körperschaften sind verpflichtet, sämtl...","Steuerpflichtige, die aufgrund ihrer Rechtsfor...",§ 7 Abs. 3 KStG 1988


In [6]:
from huggingface_hub import login
login()


In [7]:
import torch
torch.cuda.empty_cache()

In [8]:
from transformers import pipeline, BitsAndBytesConfig
import torch

quantization_config = BitsAndBytesConfig(load_in_4bit=True)

pipe = pipeline(
    "text-generation",
    model="meta-llama/Llama-3.1-8B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto",
    model_kwargs={"quantization_config": quantization_config}
)


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [9]:
print("Model loaded.")

Model loaded.


In [10]:
system_message = {
    "role": "system",
    "content": (
        "Du beantwortest Fragen zum österreichischen Steuerrecht kurz und sachlich. "
        "Erfinde keine Rechtsgrundlagen. Wenn du unsicher bist, sage das kurz."
    ),
}

def get_answer(user_text):
    messages = [
        system_message,
        {"role": "user", "content": user_text},
    ]

    output = pipe(messages, max_new_tokens=150, do_sample=False)
    return output[0]["generated_text"][-1]["content"].strip()


In [11]:
row = df.loc[0]

baseline_answer = get_answer(row["prompt"])

print("QUESTION:\n", row["prompt"])
print("\nMODEL ANSWER:\n", baseline_answer)
print("\nCORRECT ANSWER:\n", row["correct_answer"])


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION:
 Was ist die steuerliche Bemessungsgrundlage für die Körperschaftsteuer?

MODEL ANSWER:
 Die steuerliche Bemessungsgrundlage für die Körperschaftsteuer ist der Körperschaftssteuerbetrag.

CORRECT ANSWER:
 Das Einkommen, das der unbeschränkt Steuerpflichtige innerhalb eines Kalenderjahres bezogen hat (§ 7 Abs. 1 KStG).


In [12]:
sample_size = 50
sample_df = df.sample(n=min(sample_size, len(df)), random_state=42).reset_index(drop=True).copy()

baseline_answers = []

for i, row in sample_df.iterrows():
    answer = get_answer(row["prompt"])
    baseline_answers.append(answer)
    print(f"{i + 1}/{len(sample_df)} done")

sample_df["baseline_answer"] = baseline_answers
sample_df["baseline_score_0_1_2"] = ""
sample_df["baseline_note"] = ""

sample_df.to_excel("/content/baseline_results.xlsx", index=False)

sample_df[["id", "prompt", "correct_answer", "sources", "baseline_answer"]].head()


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


1/50 done


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


2/50 done


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


48/50 done


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


49/50 done
50/50 done


,id,prompt,correct_answer,sources,baseline_answer
0,SELF-070,Wann ist eine Sicherheitseinrichtung für Regis...,Seit 1. April 2017 muss jede Registrierkasse ü...,Pauschalreiseverordnung,Eine Sicherheitseinrichtung für Registrierkass...
1,SELF-030,Was ist der Teuerungsabsetzbetrag?,Eine Einmalzahlung/Absetzbetrag zur Entlastung...,SV-OG,"Der Teuerungsabsetzbetrag ist ein Betrag, der ..."
2,SELF-037,Was ist die Rechtsgrundlage für die Kapitalert...,Das EStG i.V.m. speziellen Durchführungsverord...,Dienstrechts-Novelle 2010,Die Rechtsgrundlage für die Kapitalertragsteue...
3,GRESt-AT-052,Ein Notar hat für einen Grundstückskauf die Gr...,Die Selbstberechnungserklärung ist gegenüber d...,§ 12 GrEStG 1987,Der Notar muss die Selbstberechnungserklärung ...
4,TAX-INTL-013,Ein Arbeitnehmer verdient seinen Bonus noch vo...,"Ja, laut EStR 2000 Rz 19. Einkünfte, die vor B...",EStR 2000 Rz 19,"Ja, der spätere Zufluss ist für die österreich..."


In [13]:
guided_answers = []

for i, row in sample_df.iterrows():
    user_text = f"Frage: {row['prompt']}\n\nRechtsquellen-Hinweise: {row['sources']}"
    answer = get_answer(user_text)
    guided_answers.append(answer)
    print(f"{i + 1}/{len(sample_df)} done")

sample_df["source_guided_answer"] = guided_answers
sample_df["guided_score_0_1_2"] = ""
sample_df["guided_note"] = ""

sample_df.to_excel("/content/comparison_results.xlsx", index=False)
sample_df[["id", "prompt", "sources", "baseline_answer", "source_guided_answer"]].head()

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


1/50 done


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


2/50 done


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


48/50 done


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


49/50 done
50/50 done


,id,prompt,sources,baseline_answer,source_guided_answer
0,SELF-070,Wann ist eine Sicherheitseinrichtung für Regis...,Pauschalreiseverordnung,Eine Sicherheitseinrichtung für Registrierkass...,Eine Sicherheitseinrichtung für Registrierkass...
1,SELF-030,Was ist der Teuerungsabsetzbetrag?,SV-OG,"Der Teuerungsabsetzbetrag ist ein Betrag, der ...","Der Teuerungsabsetzbetrag ist ein Betrag, der ..."
2,SELF-037,Was ist die Rechtsgrundlage für die Kapitalert...,Dienstrechts-Novelle 2010,Die Rechtsgrundlage für die Kapitalertragsteue...,"Ich bin mir nicht sicher, welche Rechtsgrundla..."
3,GRESt-AT-052,Ein Notar hat für einen Grundstückskauf die Gr...,§ 12 GrEStG 1987,Der Notar muss die Selbstberechnungserklärung ...,Die Selbstberechnungserklärung ist gegenüber d...
4,TAX-INTL-013,Ein Arbeitnehmer verdient seinen Bonus noch vo...,EStR 2000 Rz 19,"Ja, der spätere Zufluss ist für die österreich...",Die Frage nach der Relevanz des späten Zufluss...
